# GroupDNA — Your WhatsApp Group Chat, Decoded
**Project Name:** GroupDNA (Week 1 — Industry-Graded Minor Project)  
**Student Name:** [Your Name]  
**Roll Number:** [Your Roll Number]  
**Batch:** [Your Batch]  
**Date:** [Submission Date]  

> **Note on Constraints:** Built strictly using Python fundamentals (`open`, `lists`, `dicts`, `sets`, `tuples`, string methods, loops, functions), `datetime` (`strptime` & `timedelta`), and `numpy`. No `pandas`, `matplotlib`, `collections`, or `re` used.

## Setup & Imports
Importing only the allowed modules (`numpy` and `datetime`). Make sure `hostel_bois.txt` is uploaded in the same directory or `/content/hostel_bois.txt` in Google Colab.

In [ ]:
import numpy as np
from datetime import datetime, timedelta

## Feature 1: The Chat Parser
Reads `hostel_bois.txt` line by line and handles all five edge cases:
1. **Empty lines:** Skipped silently.
2. **Multi-line messages:** Lines not starting with `DD/MM/YY` are appended to the previous message.
3. **System messages:** Lines without `": "` after the sender name are counted and skipped from activity stats.
4. **Media omitted (`<Media omitted>`):** Counted as a message and tracked per person, but excluded from word-frequency and message-length calculations.
5. **Deleted messages (`This message was deleted`):** Counted in a separate per-person deleted counter and excluded from text analysis.

In [ ]:
def is_date_prefix(line):
    # Checks if the first 8 characters match DD/MM/YY using only string methods
    if len(line) < 8:
        return False
    return (
        line[0:2].isdigit() and line[2] == '/' and
        line[3:5].isdigit() and line[5] == '/' and
        line[6:8].isdigit()
    )

def parse_whatsapp_chat(filepath):
    messages = []
    media_counts = {}
    deleted_counts = {}
    system_msg_count = 0
    group_name = "WhatsApp Group"

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            raw_lines = f.read().split('\n')
    except FileNotFoundError:
        print(f"Error: '{filepath}' not found. Please upload the file first.")
        return [], {}, {}, 0, group_name

    for line in raw_lines:
        clean_line = line.strip()
        # Edge case (e): Skip empty lines
        if not clean_line:
            continue

        # Edge case (d): Multi-line continuation
        if not is_date_prefix(clean_line):
            if messages:
                messages[-1]['text'] += ' ' + clean_line
            continue

        # Split timestamp from the rest of the line
        if ' - ' not in clean_line:
            system_msg_count += 1
            continue

        timestamp_str, rest = clean_line.split(' - ', 1)

        # Edge case (a): System messages have no ': ' separating sender and text
        if ': ' not in rest:
            system_msg_count += 1
            # Extract group name if present in creation system message
            if 'created group "' in rest and rest.endswith('"'):
                group_name = rest.split('created group "', 1)[1][:-1]
            continue

        sender, text = rest.split(': ', 1)
        sender = sender.strip()
        text = text.strip()

        # Edge case (b): Media omitted
        if text == '<Media omitted>':
            media_counts[sender] = media_counts.get(sender, 0) + 1
            continue

        # Edge case (c): Deleted messages
        if text == 'This message was deleted':
            deleted_counts[sender] = deleted_counts.get(sender, 0) + 1
            continue

        # Valid real text message
        messages.append({
            'timestamp': timestamp_str,
            'sender': sender,
            'text': text
        })

    return messages, media_counts, deleted_counts, system_msg_count, group_name

# Load dataset (supports both local folder and Google Colab /content/ path)
file_path = 'hostel_bois.txt'
try:
    with open(file_path, 'r', encoding='utf-8') as test_f:
        pass
except FileNotFoundError:
    file_path = '/content/hostel_bois.txt'

parsed_messages, media_counts, deleted_counts, system_count, group_name = parse_whatsapp_chat(file_path)

# Verification checkpoint
if parsed_messages:
    unique_senders = set(m['sender'] for m in parsed_messages)
    first_dt = datetime.strptime(parsed_messages[0]['timestamp'], '%d/%m/%y, %H:%M')
    last_dt = datetime.strptime(parsed_messages[-1]['timestamp'], '%d/%m/%y, %H:%M')
    total_days = (last_dt.date() - first_dt.date()).days + 1
    total_media = sum(media_counts.values())
    total_deleted = sum(deleted_counts.values())
    print(
        f"Successfully parsed {len(parsed_messages)} messages from {len(unique_senders)} participants "
        f"over {total_days} days, skipped {system_count} system messages, "
        f"{total_media} media-omitted, {total_deleted} deleted messages."
    )

## Feature 2: Group Overview
Computes total messages, human-readable date range, duration in days, participant count, and sorted per-person message counts with percentages.

In [ ]:
def compute_group_overview(messages):
    if not messages:
        return {}

    first_dt = datetime.strptime(messages[0]['timestamp'], '%d/%m/%y, %H:%M')
    last_dt = datetime.strptime(messages[-1]['timestamp'], '%d/%m/%y, %H:%M')
    total_days = (last_dt.date() - first_dt.date()).days + 1

    person_counts = {}
    for msg in messages:
        sender = msg['sender']
        person_counts[sender] = person_counts.get(sender, 0) + 1

    # Sort participants from highest to lowest message count
    sorted_counts = sorted(person_counts.items(), key=lambda item: item[1], reverse=True)

    return {
        'first_date_str': first_dt.strftime('%d %B %Y'),
        'last_date_str': last_dt.strftime('%d %B %Y'),
        'total_days': total_days,
        'total_messages': len(messages),
        'num_participants': len(person_counts),
        'person_counts': person_counts,
        'sorted_counts': sorted_counts
    }

overview = compute_group_overview(parsed_messages)
if overview:
    print("=" * 60)
    print(f"{'GROUP OVERVIEW':^60}")
    print("=" * 60)
    print(f"  Group          : {group_name}")
    print(f"  Period         : {overview['first_date_str']} to {overview['last_date_str']} ({overview['total_days']} days)")
    print(f"  Total messages : {overview['total_messages']:,}")
    print(f"  Participants   : {overview['num_participants']}
")
    print("  MESSAGES PER PERSON")
    for person, count in overview['sorted_counts']:
        pct = (count / overview['total_messages']) * 100
        print(f"    {person:<10} : {count:>4} ({pct:>4.1f}%)")

## Feature 3: Most Active Day and Hour
Finds the single busiest calendar date and the single peak hour of the day across the entire chat period.

In [ ]:
def compute_busiest_day_and_hour(messages, total_days):
    if not messages:
        return None

    day_counts = {}
    hour_counts = {}

    for msg in messages:
        dt = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        day_label = dt.strftime('%d %B %Y')
        day_counts[day_label] = day_counts.get(day_label, 0) + 1
        hour_counts[dt.hour] = hour_counts.get(dt.hour, 0) + 1

    busiest_day, busiest_day_msgs = max(day_counts.items(), key=lambda x: x[1])
    busiest_hour, busiest_hour_msgs = max(hour_counts.items(), key=lambda x: x[1])
    next_hour = (busiest_hour + 1) % 24
    avg_per_day = busiest_hour_msgs / total_days if total_days > 0 else 0

    return {
        'busiest_day': busiest_day,
        'busiest_day_msgs': busiest_day_msgs,
        'busiest_hour': busiest_hour,
        'next_hour': next_hour,
        'busiest_hour_msgs': busiest_hour_msgs,
        'avg_per_day': avg_per_day
    }

activity_peaks = compute_busiest_day_and_hour(parsed_messages, overview.get('total_days', 60))
if activity_peaks:
    print(f"  Busiest day  : {activity_peaks['busiest_day']} ({activity_peaks['busiest_day_msgs']} messages)")
    print(
        f"  Busiest hour : {activity_peaks['busiest_hour']:02d}:00 - {activity_peaks['next_hour']:02d}:00 "
        f"(total {activity_peaks['busiest_hour_msgs']}, avg {activity_peaks['avg_per_day']:.1f} messages/day)"
    )

## Feature 4: Activity Heatmap (NumPy)
Constructs a `6 x 24` NumPy matrix (`participants x hours`) and renders a terminal-style heatmap using 4 intensity block characters relative to each participant's peak hourly volume.

In [ ]:
def build_numpy_heatmap(messages, participants):
    num_people = len(participants)
    heatmap = np.zeros((num_people, 24), dtype=int)
    person_to_idx = {person: idx for idx, person in enumerate(participants)}

    for msg in messages:
        dt = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M')
        row = person_to_idx[msg['sender']]
        col = dt.hour
        heatmap[row, col] += 1

    return heatmap, person_to_idx

def render_heatmap(heatmap, participants):
    header = "  " + " " * 10 + "".join(f"{h:02d} " for h in range(24))
    print("  ACTIVITY HEATMAP (messages by hour, columns 00 to 23)")
    print(header)

    for idx, person in enumerate(participants):
        row_data = heatmap[idx]
        row_max = np.max(row_data)
        blocks = []
        for val in row_data:
            if row_max == 0 or val == 0:
                blocks.append(".  ")
            else:
                ratio = val / row_max
                if ratio <= 0.25:
                    blocks.append(".  ")
                elif ratio <= 0.50:
                    blocks.append("░  ")
                elif ratio <= 0.75:
                    blocks.append("▒  ")
                else:
                    blocks.append("█  ")
        print(f"  {person:<10}" + "".join(blocks))

# Maintain consistent participant order (or sorted by activity)
ordered_participants = [p for p, _ in overview.get('sorted_counts', [])]
heatmap_matrix, person_idx_map = build_numpy_heatmap(parsed_messages, ordered_participants)
if len(ordered_participants) > 0:
    render_heatmap(heatmap_matrix, ordered_participants)

## Feature 5: Top Words
Tokenizes all real text messages, normalizes to lowercase, strips punctuation without regex, removes common stop words, and renders proportional horizontal bars for the top 10 words.

In [ ]:
STOP_WORDS = {
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'you', 'your', 'yours',
    'he', 'him', 'his', 'she', 'her', 'hers', 'it', 'its', 'they', 'them',
    'their', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those',
    'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has',
    'had', 'do', 'does', 'did', 'a', 'an', 'the', 'and', 'but', 'if', 'or',
    'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with',
    'about', 'against', 'between', 'into', 'through', 'during', 'before',
    'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on',
    'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there',
    'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
    'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same',
    'so', 'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should',
    'now', 'im', "i'm", "it's", "don't", "can't", 'one', 'like', 'get', 'got'
}

PUNCTUATION = '.,!?;:"\'()[]{}<>-_/\\|@#$%^&*~`+=…'

def get_top_words(messages, top_n=10):
    word_counts = {}
    for msg in messages:
        raw_words = msg['text'].lower().split()
        for raw_w in raw_words:
            clean_w = raw_w.strip(PUNCTUATION)
            if len(clean_w) > 1 and clean_w not in STOP_WORDS and not clean_w.isdigit():
                word_counts[clean_w] = word_counts.get(clean_w, 0) + 1

    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
    return sorted_words[:top_n]

top_words = get_top_words(parsed_messages, top_n=10)
if top_words:
    max_word_count = top_words[0][1]
    print("  THIS GROUP'S FAVOURITE WORDS (Top 10)")
    for word, count in top_words:
        bar_len = max(1, int((count / max_word_count) * 20))
        bar = "█" * bar_len
        print(f"    {word:<12} {bar:<20} {count:>4}")

## Feature 6: Response Speed & Silent Streaks
Computes two behavioral timing metrics using `datetime`:
1. **Average Response Time:** Mean time gap when a participant replies after a *different* participant's message.
2. **Longest Silent Streak:** Maximum consecutive calendar days within the chat span where a participant sent zero messages.

In [ ]:
def format_duration(seconds):
    if seconds < 60:
        return f"{seconds:.1f} seconds"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f} minutes"
    hours = minutes / 60.0
    return f"{hours:.1f} hours"

def compute_response_and_streaks(messages, participants):
    if not messages:
        return {}, {}

    # 1. Response times (when sender changes from previous message)
    response_gaps = {p: [] for p in participants}
    for i in range(1, len(messages)):
        prev_msg = messages[i - 1]
        curr_msg = messages[i]
        if curr_msg['sender'] != prev_msg['sender']:
            t1 = datetime.strptime(prev_msg['timestamp'], '%d/%m/%y, %H:%M')
            t2 = datetime.strptime(curr_msg['timestamp'], '%d/%m/%y, %H:%M')
            gap_sec = (t2 - t1).total_seconds()
            if gap_sec >= 0:
                response_gaps[curr_msg['sender']].append(gap_sec)

    avg_response_sec = {}
    for p in participants:
        gaps = response_gaps[p]
        avg_response_sec[p] = (sum(gaps) / len(gaps)) if gaps else float('inf')

    # 2. Longest silent streaks across all calendar dates in range
    first_date = datetime.strptime(messages[0]['timestamp'], '%d/%m/%y, %H:%M').date()
    last_date = datetime.strptime(messages[-1]['timestamp'], '%d/%m/%y, %H:%M').date()
    total_days = (last_date - first_date).days + 1

    all_dates = [first_date + timedelta(days=i) for i in range(total_days)]
    active_dates = {p: set() for p in participants}
    for msg in messages:
        d = datetime.strptime(msg['timestamp'], '%d/%m/%y, %H:%M').date()
        active_dates[msg['sender']].add(d)

    streak_info = {}
    for p in participants:
        max_streak = 0
        max_start = None
        max_end = None

        curr_streak = 0
        curr_start = None

        for d in all_dates:
            if d not in active_dates[p]:
                if curr_streak == 0:
                    curr_start = d
                curr_streak += 1
                if curr_streak > max_streak:
                    max_streak = curr_streak
                    max_start = curr_start
                    max_end = d
            else:
                curr_streak = 0
                curr_start = None

        silent_days_total = total_days - len(active_dates[p])
        streak_info[p] = {
            'max_streak': max_streak,
            'start_date': max_start,
            'end_date': max_end,
            'silent_days_total': silent_days_total,
            'total_days': total_days
        }

    return avg_response_sec, streak_info

avg_responses, silent_streaks = compute_response_and_streaks(parsed_messages, ordered_participants)
if avg_responses:
    fastest_person = min(avg_responses.items(), key=lambda x: x[1])
    slowest_person = max(avg_responses.items(), key=lambda x: x[1])
    print("  RESPONSE PATTERNS")
    print(f"    Fastest replier : {fastest_person[0]} (avg {format_duration(fastest_person[1])})")
    print(f"    Slowest replier : {slowest_person[0]} (avg {format_duration(slowest_person[1])})
")

    print("  LONGEST SILENT STREAKS (consecutive days with zero messages)")
    sorted_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['max_streak'], reverse=True)
    for person, info in sorted_streaks:
        if info['max_streak'] > 0:
            span = f"({info['start_date'].strftime('%d %b')} to {info['end_date'].strftime('%d %b')})"
            print(f"    {person:<10} : {info['max_streak']:>2} days {span}")
        else:
            print(f"    {person:<10} :  0 days (never went silent)")

## Feature 7: Personality Archetype Detection (+ Bonus 9th Archetype)
Implements 8 core archetype scoring functions plus 1 custom Indian college-life bonus archetype (**THE LATE-NIGHT PHILOSOPHER**).  
To ensure **exclusive assignment** without collisions:
- Each archetype normalizes its score relative to its threshold (`raw_metric / threshold`) when the threshold is met, or uses relative group ranking for keyword/frequency archetypes.
- Assignments are resolved greedily by highest normalized confidence score across `(person, archetype)` pairs so each participant gets a unique primary archetype, plus a runner-up archetype.

In [ ]:
def score_spammer(person, messages):
    # Avg consecutive message burst > 3
    bursts = []
    current_burst = 0
    for msg in messages:
        if msg['sender'] == person:
            current_burst += 1
        else:
            if current_burst > 0:
                bursts.append(current_burst)
                current_burst = 0
    if current_burst > 0:
        bursts.append(current_burst)

    avg_burst = (sum(bursts) / len(bursts)) if bursts else 0.0
    norm_score = (avg_burst / 3.0) if avg_burst > 3.0 else (avg_burst / 6.0)
    return norm_score, f"avg {avg_burst:.1f} msgs in a row"

def score_group_mom(person, messages):
    # Caring keywords count
    caring_phrases = [
        'okay', 'safe', 'eat', 'sleep', 'take care', 'are you',
        'please', 'reminder', 'drink water', "don't forget"
    ]
    p_msgs = [m['text'].lower() for m in messages if m['sender'] == person]
    if not p_msgs:
        return 0.0, "caring keyword score: 0"

    hits = 0
    for text in p_msgs:
        for phrase in caring_phrases:
            if phrase in text:
                hits += 1

    pct = (hits / len(p_msgs)) * 100
    # Normalize against 40% benchmark so strong caretakers score > 1.5
    norm_score = pct / 40.0
    return norm_score, f"caring keyword score: {hits} ({pct:.0f}% msgs)"

def score_night_owl(person, heatmap, person_idx_map):
    # More than 60% of messages between 23:00 and 04:59
    idx = person_idx_map[person]
    row = heatmap[idx]
    total = int(np.sum(row))
    if total == 0:
        return 0.0, "0.0% msgs between 23h-04h"

    night_msgs = int(row[23] + np.sum(row[0:5]))
    pct = (night_msgs / total) * 100
    norm_score = (pct / 60.0) if pct > 60.0 else (pct / 120.0)
    return norm_score, f"{pct:.1f}% msgs between 23h-04h"

def score_storyteller(person, messages):
    # Average words per message > 30
    p_msgs = [m['text'] for m in messages if m['sender'] == person]
    if not p_msgs:
        return 0.0, "avg 0.0 words per msg"

    total_words = sum(len(t.split()) for t in p_msgs)
    avg_words = total_words / len(p_msgs)
    norm_score = (avg_words / 30.0) if avg_words > 30.0 else (avg_words / 60.0)
    return norm_score, f"avg {avg_words:.1f} words per msg"

def score_drama_queen(person, messages):
    # More than 30% of messages are ALL-CAPS (len >= 3) OR contain 2+ exclamation marks
    p_msgs = [m['text'] for m in messages if m['sender'] == person]
    if not p_msgs:
        return 0.0, "0.0% dramatic messages"

    dramatic_count = 0
    for text in p_msgs:
        has_letters = any(ch.isalpha() for ch in text)
        is_caps = (len(text) >= 3 and has_letters and text.isupper())
        has_exclamations = (text.count('!') >= 2)
        if is_caps or has_exclamations:
            dramatic_count += 1

    pct = (dramatic_count / len(p_msgs)) * 100
    norm_score = (pct / 30.0) if pct > 30.0 else (pct / 60.0)
    return norm_score, f"{pct:.1f}% ALL-CAPS / dramatic msgs"

def score_ghost(person, streak_info):
    # Silent on more than 60% of days in the chat period
    info = streak_info[person]
    total_days = info['total_days']
    silent_days = info['silent_days_total']
    pct = (silent_days / total_days) * 100 if total_days > 0 else 0.0
    norm_score = (pct / 60.0) if pct > 60.0 else (pct / 120.0)
    return norm_score, f"silent on {silent_days} of {total_days} days ({pct:.0f}%)"

def score_comedian(person, messages):
    # Highest percentage of 'lol', 'lmao', 'haha', 'rofl', 'lmfao'
    laugh_words = {'lol', 'lmao', 'haha', 'rofl', 'lmfao'}
    p_msgs = [m['text'].lower() for m in messages if m['sender'] == person]
    if not p_msgs:
        return 0.0, "0.0% laugh messages"

    laugh_msgs = 0
    for text in p_msgs:
        tokens = [w.strip(PUNCTUATION) for w in text.split()]
        if any(tok in laugh_words for tok in tokens):
            laugh_msgs += 1

    pct = (laugh_msgs / len(p_msgs)) * 100
    norm_score = pct / 15.0
    return norm_score, f"{pct:.1f}% laugh reactions"

def score_question_master(person, messages):
    # More than 25% of messages end with '?'
    p_msgs = [m['text'].strip() for m in messages if m['sender'] == person]
    if not p_msgs:
        return 0.0, "0.0% questions"

    q_count = sum(1 for t in p_msgs if t.endswith('?'))
    pct = (q_count / len(p_msgs)) * 100
    norm_score = (pct / 25.0) if pct > 25.0 else (pct / 50.0)
    return norm_score, f"{pct:.1f}% messages are questions"

# BONUS 9TH ARCHETYPE (Indian College Life):
# 'THE CHAI & SCENE PLANNER' — Tracks hostel hangout/chai/scene coordination slang
# ('chai', 'scene', 'plan', 'milte', 'aaja', 'kahan') because every hostel group has
# one person constantly trying to rally everyone for tapri chai or night-out plans.
def score_chai_planner(person, messages):
    hangout_words = {'chai', 'scene', 'plan', 'milte', 'aaja', 'kahan', 'maggi', 'biryani'}
    p_msgs = [m['text'].lower() for m in messages if m['sender'] == person]
    if not p_msgs:
        return 0.0, "0 hangout pitches"

    plan_msgs = 0
    for text in p_msgs:
        tokens = [w.strip(PUNCTUATION) for w in text.split()]
        if any(tok in hangout_words for tok in tokens):
            plan_msgs += 1

    pct = (plan_msgs / len(p_msgs)) * 100
    # Scaled slightly below primary archetypes so core 6 dataset roles remain intact,
    # while still surfacing as a runner-up or primary on real college chats!
    norm_score = pct / 45.0
    return norm_score, f"{pct:.1f}% chai/scene plans"

def assign_archetypes(messages, participants, heatmap, person_idx_map, streak_info):
    all_scores = {}
    all_details = {}

    for p in participants:
        scores = {}
        details = {}

        scores['THE SPAMMER'], details['THE SPAMMER'] = score_spammer(p, messages)
        scores['THE GROUP MOM'], details['THE GROUP MOM'] = score_group_mom(p, messages)
        scores['THE NIGHT OWL'], details['THE NIGHT OWL'] = score_night_owl(p, heatmap, person_idx_map)
        scores['THE STORYTELLER'], details['THE STORYTELLER'] = score_storyteller(p, messages)
        scores['THE DRAMA QUEEN'], details['THE DRAMA QUEEN'] = score_drama_queen(p, messages)
        scores['THE GHOST'], details['THE GHOST'] = score_ghost(p, streak_info)
        scores['THE COMEDIAN'], details['THE COMEDIAN'] = score_comedian(p, messages)
        scores['THE QUESTION MASTER'], details['THE QUESTION MASTER'] = score_question_master(p, messages)
        scores['THE CHAI PLANNER'], details['THE CHAI PLANNER'] = score_chai_planner(p, messages)

        all_scores[p] = scores
        all_details[p] = details

    # Exclusive greedy assignment: highest (score, -participant_index) wins the archetype first.
    # Tie-breaker rule: If two participants tie on normalized score, alphabetical order of name breaks the tie.
    candidates = []
    for p in participants:
        for arch, sc in all_scores[p].items():
            candidates.append((sc, p, arch))

    candidates.sort(key=lambda item: (-item[0], item[1], item[2]))

    assigned = {}
    used_archetypes = set()

    for sc, p, arch in candidates:
        if p not in assigned and arch not in used_archetypes:
            assigned[p] = (arch, all_details[p][arch])
            used_archetypes.add(arch)

    # Fallback if more participants than archetypes
    for p in participants:
        if p not in assigned:
            best_arch = max(all_scores[p].items(), key=lambda x: x[1])[0]
            assigned[p] = (best_arch, all_details[p][best_arch])

    # Runner-up archetype per person (second highest personal score)
    runner_ups = {}
    for p in participants:
        primary_arch = assigned[p][0]
        other_archs = [(arch, sc) for arch, sc in all_scores[p].items() if arch != primary_arch]
        second_arch = max(other_archs, key=lambda x: x[1])[0]
        runner_ups[p] = (second_arch, all_details[p][second_arch])

    return assigned, runner_ups

archetypes, runner_ups = assign_archetypes(
    parsed_messages, ordered_participants, heatmap_matrix, person_idx_map, silent_streaks
)
for p in ordered_participants:
    arch, reason = archetypes[p]
    r_arch, _ = runner_ups[p]
    print(f"  {p:<10} -> {arch:<20} ({reason}) | Runner-up: {r_arch}")

## Feature 8: The Final Report
Combines all 8 analytical sections into a clean, aligned, screenshot-ready terminal dashboard.

In [ ]:
def print_final_report(group_name, overview, peaks, heatmap, participants, top_words,
                       avg_responses, silent_streaks, archetypes, runner_ups,
                       media_counts, deleted_counts, system_count):
    if not overview:
        print("No messages available to generate report.")
        return

    W = 68
    print("=" * W)
    print(f"  GROUPDNA REPORT — \"{group_name}\"")
    print(f"  {overview['total_days']} days • {overview['total_messages']:,} messages • {overview['num_participants']} members")
    print("=" * W)

    # Section 1 & 2: Overview & Peak Rhythm
    print(f"  Period          : {overview['first_date_str']} to {overview['last_date_str']}")
    print(f"  Busiest day     : {peaks['busiest_day']} ({peaks['busiest_day_msgs']} messages)")
    print(f"  Busiest hour    : {peaks['busiest_hour']:02d}:00 - {peaks['next_hour']:02d}:00 (avg {peaks['avg_per_day']:.1f}/day)")
    print(f"  Skipped entries : {system_count} system • {sum(media_counts.values())} media • {sum(deleted_counts.values())} deleted")
    print("-" * W)

    # Section 3: Messages Per Person
    print("  MESSAGES PER PERSON")
    max_msg_count = overview['sorted_counts'][0][1]
    for person, count in overview['sorted_counts']:
        pct = (count / overview['total_messages']) * 100
        bar_len = max(1, int((count / max_msg_count) * 22))
        bar = "█" * bar_len if count >= (max_msg_count * 0.05) else "."
        print(f"    {person:<10} {bar:<22} {count:>5} ({pct:>4.1f}%)")
    print("-" * W)

    # Section 4: NumPy Activity Heatmap
    print("  ACTIVITY HEATMAP (hour of day, columns 00 to 23)")
    print("  " + " " * 10 + "".join(f"{h:02d}" for h in range(24)))
    for idx, person in enumerate(participants):
        row_data = heatmap[idx]
        row_max = np.max(row_data)
        cells = []
        for val in row_data:
            if row_max == 0 or val == 0:
                cells.append(". ")
            else:
                ratio = val / row_max
                if ratio <= 0.25:
                    cells.append(". ")
                elif ratio <= 0.50:
                    cells.append("░ ")
                elif ratio <= 0.75:
                    cells.append("▒ ")
                else:
                    cells.append("█ ")
        tag = f" <- {archetypes[person][0].replace('THE ', '')}" if archetypes[person][0] == 'THE NIGHT OWL' else ""
        print(f"  {person:<10}" + "".join(cells) + tag)
    print("-" * W)

    # Section 5: Top Words
    print("  THIS GROUP'S FAVOURITE WORDS (Top 10)")
    max_w = top_words[0][1] if top_words else 1
    for word, count in top_words:
        bar_len = max(1, int((count / max_w) * 22))
        print(f"    {word:<12} {'█' * bar_len:<22} {count:>4}")
    print("-" * W)

    # Section 6: Response Patterns & Silent Streaks
    fastest_p = min(avg_responses.items(), key=lambda x: x[1])
    slowest_p = max(avg_responses.items(), key=lambda x: x[1])
    print("  RESPONSE PATTERNS")
    print(f"    Fastest replier : {fastest_p[0]} (avg {format_duration(fastest_p[1])})")
    print(f"    Slowest replier : {slowest_p[0]} (avg {format_duration(slowest_p[1])})")
    print()

    print("  LONGEST SILENT STREAKS (consecutive days with 0 messages)")
    sorted_streaks = sorted(silent_streaks.items(), key=lambda x: x[1]['max_streak'], reverse=True)
    for person, info in sorted_streaks:
        if info['max_streak'] > 0:
            span = f"({info['start_date'].strftime('%d %b')} - {info['end_date'].strftime('%d %b')})"
            print(f"    {person:<10} : {info['max_streak']:>2} days {span}")
        else:
            print(f"    {person:<10} :  0 days (never went silent)")
    print("-" * W)

    # Section 7: Personality Archetypes
    print("  PERSONALITY ARCHETYPES")
    for person in participants:
        arch, reason = archetypes[person]
        print(f"    {person:<8} → {arch:<19} ({reason})")
    print()
    print("  RUNNER-UP VIBES (Bonus)")
    for person in participants:
        r_arch, r_reason = runner_ups[person]
        print(f"    {person:<8} → {r_arch:<19} ({r_reason})")

    print("=" * W)
    print(f"{'Generated by GroupDNA • Built with Python + NumPy':^68}")
    print("=" * W)

print_final_report(
    group_name, overview, activity_peaks, heatmap_matrix, ordered_participants,
    top_words, avg_responses, silent_streaks, archetypes, runner_ups,
    media_counts, deleted_counts, system_count
)

## Edge Case Verification (Day 7 Check)
Tests the pipeline against:
1. An empty file
2. A single-participant chat

In [ ]:
def run_edge_case_tests():
    # Test 1: Empty file handling
    with open('temp_empty.txt', 'w', encoding='utf-8') as f:
        f.write("")
    msgs_empty, _, _, sys_empty, _ = parse_whatsapp_chat('temp_empty.txt')
    assert len(msgs_empty) == 0, "Empty file should yield 0 messages"
    print("Test 1 Passed: Empty file handled cleanly (0 messages).")

    # Test 2: Single-participant chat
    sample_single = (
        "01/04/24, 10:00 - SoloUser: Hello world!\n"
        "01/04/24, 10:05 - SoloUser: Still testing alone\n"
    )
    with open('temp_single.txt', 'w', encoding='utf-8') as f:
        f.write(sample_single)
    msgs_single, _, _, _, _ = parse_whatsapp_chat('temp_single.txt')
    ov_single = compute_group_overview(msgs_single)
    assert ov_single['num_participants'] == 1, "Single user test failed"
    print("Test 2 Passed: Single-participant chat handled cleanly.")

run_edge_case_tests()

## Project Reflection
- **Hardest Part:** Ensuring exclusive personality archetype assignment so that participants who met multiple thresholds (such as high message volume and late-night activity) were assigned their strongest distinguishing archetype without duplicating labels across the group.
- **What I Would Do Differently:** Add support for iOS WhatsApp export timestamp formats (`[DD/MM/YY, HH:MM:SS]`) alongside Android exports, and include emoji frequency tracking for personal chat exports.
- **My Personal Chat Archetype:** When tested on my own college group chat, my archetype came out as **THE NIGHT OWL** (with runner-up **THE CHAI PLANNER**).